# Sequence Completion (GPT)

In [2]:
model_names = ["openai-community/gpt2-large",
               "openai-community/gpt2-xl",
               "facebook/opt-1.3b",
               "tiiuae/Falcon3-1B-Instruct"]
pretrained_model = model_names[0]

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
model = AutoModelForCausalLM.from_pretrained(pretrained_model).to(device)

import torchinfo
summary = torchinfo.summary(model, 
                  input_data=torch.randint(0, tokenizer.vocab_size, (32, 64), device=device),
                  col_names=["input_size", "output_size", "num_params"])
print(summary)

# Define tasks and example inputs
tasks = {
    "story completion": "In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley in the Andes Mountains.",
    "question answering": "Q: Who wrote the book the origin of species? A:",
    "translation": "Translate English to French: I'm home = Je suis là | I love you = ",
    "reading comprehension": "Tom goes everywhere with Catherine Green, a 54-year-old secretary. Catherine and Tom live in Sweden. Q: How old is Catherine? A: 54 | Q: Where does she live? A:"
}

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=64,
            do_sample=True,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Run each task
for task, input_text in tasks.items():
    print(f"\n--- {task.upper()} ---")
    print("Input:", input_text)
    output = generate(input_text)
    # Remove input prefix from the output
    output = output[len(input_text):].strip()
    print("Output:", output)


Using device: cuda
Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
GPT2LMHeadModel                                    [32, 64]                  [32, 20, 64, 64]          --
├─GPT2Model: 1-1                                   [32, 64]                  [32, 20, 64, 64]          --
│    └─Embedding: 2-1                              [32, 64]                  [32, 64, 1280]            64,328,960
│    └─Embedding: 2-2                              [1, 64]                   [1, 64, 1280]             1,310,720
│    └─Dropout: 2-3                                [32, 64, 1280]            [32, 64, 1280]            --
│    └─ModuleList: 2-4                             --                        --                        --
│    │    └─GPT2Block: 3-1                         [32, 64, 1280]            [32, 64, 1280]            19,677,440
│    │    └─GPT2Block: 3-2                         [32, 64, 1280]            [32, 64, 1280]            19

# Sequence Transformation (T5)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load a larger pretrained model for better performance
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(device)

import torchinfo
# Display model architecture
summary = torchinfo.summary(model, 
                  input_data=(torch.randint(0, tokenizer.vocab_size, (32, 64), device=device),
                              torch.ones((32, 64), dtype=torch.int, device=device),
                              torch.randint(0, tokenizer.vocab_size, (32, 64), device=device)),
                  col_names=["input_size", "output_size", "num_params"])
print(summary)

# Define tasks and example inputs
tasks = {
    "summarization": "summarize: The quick brown fox jumps over the lazy dog. The dog did not mind and kept sleeping.",
    "translation": "translate English to German: I love programming and natural language processing.",
    "paraphrasing": "paraphrase: The movie was incredibly boring, but I liked the music.",
    "grammar correction": "grammar correction: She no went to the market yesterday.",
    "question generation": "generate question: Albert Einstein was born in Germany in 1879."
}

def generate(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Run each task
for task, input_text in tasks.items():
    print(f"\n--- {task.upper()} ---")
    print("Input:", input_text)
    print("Output:", generate(input_text))


Using device: cuda


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
T5ForConditionalGeneration                                   [32, 64]                  [32, 64, 1024]            --
├─T5Stack: 1-1                                               --                        [32, 64, 1024]            341,231,104
├─T5Stack: 1-2                                               --                        --                        (recursive)
│    └─Embedding: 2-1                                        [32, 64]                  [32, 64, 1024]            32,899,072
├─T5Stack: 1-3                                               --                        --                        (recursive)
│    └─Dropout: 2-2                                          [32, 64, 1024]            [32, 64, 1024]            --
│    └─ModuleList: 2-3                                       --                        --                        --
│    │    └─T5Block: 3-1        